# Geometric Field Modulation for Ternary Parameter Golf

Phase 0: Measure ternary quantization error structure (DECISION GATE).
If structure exists → proceed with geometric field G experiments.
If no structure → stop and write up negative result.

In [ ]:
# === 1. SETUP ===
import shutil, os, glob

from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/parameter-golf'

OFFICIAL = '/content/parameter-golf'
AUX = '/content/parameter-golf-aux'
WORK = '/content/ternary_work'
TERNARY_SRC = f'{OFFICIAL}/records/track_10min_16mb/2026-03-24_74M_Ternary_UNet_FP8_10L_8192BPE_YaRN_NeoMuon'

# Clone repos
%cd /content
if not os.path.exists(OFFICIAL):
    !git clone --depth 1 https://github.com/openai/parameter-golf.git
else:
    %cd {OFFICIAL}
    !git pull
if not os.path.exists(AUX):
    !git clone https://github.com/jamesconde/parameter-golf-aux.git
else:
    %cd {AUX}
    !git pull

!pip install -q sentencepiece huggingface-hub datasets tqdm zstandard

# Create working directory
os.makedirs(WORK, exist_ok=True)
os.makedirs(f'{WORK}/geometric_field', exist_ok=True)
os.makedirs(f'{WORK}/data/tokenizers', exist_ok=True)

# Copy ternary training script + tokenizer
shutil.copy2(f'{TERNARY_SRC}/train_gpt_cuda_ternary.py', f'{WORK}/train_gpt_cuda_ternary.py')
shutil.copy2(f'{TERNARY_SRC}/fineweb_8192_bpe.model', f'{WORK}/fineweb_8192_bpe.model')
shutil.copy2(f'{TERNARY_SRC}/fineweb_8192_bpe.vocab', f'{WORK}/fineweb_8192_bpe.vocab')
shutil.copy2(f'{TERNARY_SRC}/fineweb_8192_bpe.model', f'{WORK}/data/tokenizers/fineweb_8192_bpe.model')

# Copy our analysis scripts
for f in glob.glob(f'{AUX}/geometric_field/*.py'):
    shutil.copy2(f, f'{WORK}/geometric_field/{os.path.basename(f)}')

print(f'Working directory: {WORK}')

In [ ]:
# === 2. DOWNLOAD SP8192 DATA (from HuggingFace) ===
%cd {WORK}

data_dir = f'{WORK}/data/datasets/fineweb10B_sp8192'

if os.path.exists(data_dir) and glob.glob(f'{data_dir}/fineweb_val_*.bin'):
    print(f'Data already exists at {data_dir}')
else:
    print('Downloading sp8192 data from HuggingFace...')
    from huggingface_hub import snapshot_download

    # Try as model repo first (the setup.sh uses 'hf download' without --repo-type)
    for repo_type in [None, "model", "dataset"]:
        try:
            rt_label = repo_type or "default"
            print(f'  Trying repo_type={rt_label}...')
            snapshot_download(
                repo_id="sproos/parameter-golf-tokenizers",
                repo_type=repo_type,
                allow_patterns=["datasets/fineweb10B_sp8192/*"],
                local_dir="./data",
            )
            break
        except Exception as e:
            print(f'  Failed: {str(e)[:100]}')
    else:
        # Fallback: use the official parameter-golf downloader to tokenize from sp1024
        print('  All HF downloads failed.')
        print('  Falling back: download sp1024 data and retokenize...')
        # Actually, we can use the official cached_challenge_fineweb.py if sp8192 is available
        %cd {OFFICIAL}
        !python3 data/cached_challenge_fineweb.py --variant sp8192 --train-shards 1 || true
        %cd {WORK}
        # Symlink if it was downloaded there
        src = f'{OFFICIAL}/data/datasets/fineweb10B_sp8192'
        if os.path.exists(src) and not os.path.exists(data_dir):
            os.makedirs(os.path.dirname(data_dir), exist_ok=True)
            os.symlink(src, data_dir)

train_files = glob.glob(f'{data_dir}/fineweb_train_*.bin')
val_files = glob.glob(f'{data_dir}/fineweb_val_*.bin')
print(f'Train shards: {len(train_files)}, Val shards: {len(val_files)}')
if not val_files:
    print('\nERROR: No data downloaded. You may need to:')
    print('1. Get a HuggingFace token: https://huggingface.co/settings/tokens')
    print('2. Run in a cell: from huggingface_hub import login; login()')
    print('3. Re-run this cell')

In [ ]:
# === 3. PATCH TERNARY SCRIPT FOR NON-HOPPER GPU ===
%cd {WORK}

!python3 geometric_field/patch_ternary.py train_gpt_cuda_ternary.py

# Verify
!python3 -c "import py_compile; py_compile.compile('train_gpt_cuda_ternary.py', doraise=True); print('Syntax OK')"

import torch
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} ({vram_gb:.0f}GB)')

In [ ]:
# === 4. TRAIN TERNARY BASELINE (~1 hour) ===
# Patch saves raw state_dict as final_model_raw_sd.pt before quantization
%cd {WORK}

import os, shutil, torch

# Auto-detect batch size based on GPU VRAM
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
if vram_gb >= 70:
    BATCH = 524288   # 80GB+ GPU (original submission setting)
elif vram_gb >= 35:
    BATCH = 131072   # 40GB A100 (reduced to fit)
else:
    BATCH = 65536    # Smaller GPUs
print(f'GPU VRAM: {vram_gb:.0f}GB → TRAIN_BATCH_TOKENS={BATCH}')

if os.path.exists('final_model_raw_sd.pt'):
    print(f'Raw state_dict exists ({os.path.getsize("final_model_raw_sd.pt")/1e6:.1f} MB)')
elif os.path.exists(f'{DRIVE_DIR}/ternary/final_model_raw_sd.pt'):
    shutil.copy2(f'{DRIVE_DIR}/ternary/final_model_raw_sd.pt', 'final_model_raw_sd.pt')
    print(f'Loaded from Drive ({os.path.getsize("final_model_raw_sd.pt")/1e6:.1f} MB)')
else:
    print('Training ternary baseline (~1 hour)...')
    !RUN_ID=ternary_baseline \
     DATA_PATH=./data/datasets/fineweb10B_sp8192 \
     TOKENIZER_PATH=./data/tokenizers/fineweb_8192_bpe.model \
     VOCAB_SIZE=8192 \
     NUM_LAYERS=10 MODEL_DIM=768 NUM_HEADS=8 NUM_KV_HEADS=4 MLP_MULT=4 \
     EMBED_DIM=254 \
     BITNET_GROUP_SIZE=128 \
     ACTIVATION=relu2 \
     ROPE_TYPE=yarn YARN_MAX_LEN=2048 ROPE_BASE=5000 \
     LOGIT_SOFTCAP=10 SOFTCAP_TYPE=poly \
     QK_GAIN_INIT=2.25 \
     MATRIX_OPTIMIZER=muon MUON_BACKEND_STEPS=3 \
     MUON_MOMENTUM=0.95 MUON_MOMENTUM_WARMUP_START=0.85 MUON_MOMENTUM_WARMUP_STEPS=500 \
     MUON_WD=0.0 ADAM_WD=0.05 ADAM_LR=0.05 \
     MATRIX_LR=0.04 SCALAR_LR=0.02 TIED_EMBED_LR=0.02 HEAD_LR=0.02 \
     WARMDOWN_FRACTION=0.2 \
     FP_STORAGE=FP8 \
     TRAIN_BATCH_TOKENS={BATCH} TRAIN_SEQ_LEN=1024 \
     BIGRAM_HASH=0 SMEAR=0 DIFF_ATTN=0 MLP_GROUPS=0 MTP_HEADS=0 \
     TIE_EMBEDDINGS=1 \
     ITERATIONS=10000 MAX_WALLCLOCK_SECONDS=3600 \
     WARMUP_STEPS=5 VAL_LOSS_EVERY=500 TRAIN_LOG_EVERY=100 \
     USE_COMPILE=0 \
     SEED=42 \
     python3 train_gpt_cuda_ternary.py

In [ ]:
# === 5. SAVE CHECKPOINTS TO DRIVE ===
import shutil, os
os.makedirs(f'{DRIVE_DIR}/ternary', exist_ok=True)

for f in ['final_model_raw_sd.pt', 'final_model.ternary.ptz']:
    if os.path.exists(f):
        shutil.copy2(f, f'{DRIVE_DIR}/ternary/{f}')
        print(f'Saved {f} ({os.path.getsize(f)/1e6:.1f} MB) → Drive')
    else:
        print(f'{f} not found')

In [ ]:
# === 6. PHASE 0: QUANTIZATION ERROR STRUCTURE ANALYSIS (DECISION GATE) ===
%cd {WORK}

!python3 geometric_field/phase0_analysis.py \
    --checkpoint final_model_raw_sd.pt \
    --model-dim 768 --num-heads 8 --num-kv-heads 4 --mlp-mult 4 --num-layers 10 \
    --group-size 128 \
    --output geometric_field/phase0_results.json

In [ ]:
# === 7. SAVE RESULTS + DISPLAY CLASSIFICATION ===
import shutil, json, os

for f in ['geometric_field/phase0_results.json', 'geometric_field/phase0_results_profiles.pt']:
    if os.path.exists(f):
        os.makedirs(f'{DRIVE_DIR}/ternary/results', exist_ok=True)
        shutil.copy2(f, f'{DRIVE_DIR}/ternary/results/{os.path.basename(f)}')

if os.path.exists('geometric_field/phase0_results.json'):
    with open('geometric_field/phase0_results.json') as f:
        results = json.load(f)
    c = results['classification']
    print('=' * 50)
    print(f"Scenario: {c['scenario']}")
    print(f"PROCEED: {c['proceed']}")
    print(f"Row structure:  {c['mean_row_structure']:.2f} (threshold: 2.0)")
    print(f"Col structure:  {c['mean_col_structure']:.2f} (threshold: 2.0)")
    print(f"Layer variation: {c['layer_variation']:.3f} (threshold: 0.15)")
    print('=' * 50)
    if c['proceed']:
        print('\n-> Column structure detected. Proceed with G experiments.')
    else:
        print('\n-> No exploitable structure. Write up as negative result.')
else:
    print('Phase 0 results not found. Run the analysis cell first.')

In [ ]:
# === 8. COMPUTE G SIGNALS (Experiment 1) ===
# Word-boundary direction + input covariance — tells G which columns to compress
%cd {WORK}

!python3 geometric_field/compute_signals.py \
    --checkpoint final_model_raw_sd.pt \
    --tokenizer ./data/tokenizers/fineweb_8192_bpe.model \
    --data-path ./data/datasets/fineweb10B_sp8192 \
    --output geometric_field/g_signals.pt \
    --max-val-tokens 500000

# Save to Drive
import shutil, os
if os.path.exists('geometric_field/g_signals.pt'):
    os.makedirs(f'{DRIVE_DIR}/ternary/results', exist_ok=True)
    shutil.copy2('geometric_field/g_signals.pt', f'{DRIVE_DIR}/ternary/results/g_signals.pt')
    print('Signals saved to Drive')

In [ ]:
# === 9. ABLATION GRID (Experiment 4) ===
# Train with G at different alpha/beta values and compare BPB
%cd {WORK}

import os, torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 524288 if vram_gb >= 70 else (131072 if vram_gb >= 35 else 65536)

# Ablation runs: control + covariance-only + boundary-only + combined
experiments = [
    ("control_1", "0.0", "0.0"),
    ("control_2", "0.0", "0.0"),
    ("cov_01",    "0.1", "0.0"),
    ("cov_03",    "0.3", "0.0"),
    ("cov_05",    "0.5", "0.0"),
    ("bnd_01",    "0.0", "0.1"),
    ("bnd_03",    "0.0", "0.3"),
    ("bnd_05",    "0.0", "0.5"),
    ("both_01",   "0.1", "0.1"),
    ("both_03",   "0.3", "0.3"),
    ("both_05",   "0.5", "0.5"),
]

for name, alpha, beta in experiments:
    log_file = f'logs/geom_{name}.txt'
    if os.path.exists(log_file):
        # Check if run completed
        with open(log_file) as f:
            content = f.read()
        if 'val_bpb' in content and 'stopping_early' in content:
            print(f'SKIP {name} — already complete')
            continue

    print(f'\n{"="*60}')
    print(f'Running {name}: alpha={alpha}, beta={beta}')
    print(f'{"="*60}')

    !RUN_ID=geom_{name} \
     DATA_PATH=./data/datasets/fineweb10B_sp8192 \
     TOKENIZER_PATH=./data/tokenizers/fineweb_8192_bpe.model \
     VOCAB_SIZE=8192 \
     NUM_LAYERS=10 MODEL_DIM=768 NUM_HEADS=8 NUM_KV_HEADS=4 MLP_MULT=4 \
     EMBED_DIM=254 \
     BITNET_GROUP_SIZE=128 \
     ACTIVATION=relu2 \
     ROPE_TYPE=yarn YARN_MAX_LEN=2048 ROPE_BASE=5000 \
     LOGIT_SOFTCAP=10 SOFTCAP_TYPE=poly \
     QK_GAIN_INIT=2.25 \
     MATRIX_OPTIMIZER=muon MUON_BACKEND_STEPS=3 \
     MUON_MOMENTUM=0.95 MUON_MOMENTUM_WARMUP_START=0.85 MUON_MOMENTUM_WARMUP_STEPS=500 \
     MUON_WD=0.0 ADAM_WD=0.05 ADAM_LR=0.05 \
     MATRIX_LR=0.04 SCALAR_LR=0.02 TIED_EMBED_LR=0.02 HEAD_LR=0.02 \
     WARMDOWN_FRACTION=0.2 \
     FP_STORAGE=FP8 \
     TRAIN_BATCH_TOKENS={BATCH} TRAIN_SEQ_LEN=1024 \
     BIGRAM_HASH=0 SMEAR=0 DIFF_ATTN=0 MLP_GROUPS=0 MTP_HEADS=0 \
     TIE_EMBEDDINGS=1 \
     ITERATIONS=10000 MAX_WALLCLOCK_SECONDS=3600 \
     WARMUP_STEPS=5 VAL_LOSS_EVERY=500 TRAIN_LOG_EVERY=100 \
     USE_COMPILE=0 \
     GEOM_ALPHA={alpha} GEOM_BETA={beta} \
     GEOM_SIGNALS=geometric_field/g_signals.pt \
     SEED=42 \
     python3 train_gpt_cuda_ternary.py

In [ ]:
# === 10. COLLECT AND COMPARE RESULTS ===
%cd {WORK}
import re, glob

results = []
for logfile in sorted(glob.glob('logs/geom_*.txt') + glob.glob('logs/cuda/geom_*.txt')):
    name = os.path.basename(logfile).replace('.txt', '')
    with open(logfile) as f:
        text = f.read()
    # Find final sliding window BPB
    m = re.search(r'final_int6_sliding_window_exact\s+val_loss:(\d+\.\d+)\s+val_bpb:(\d+\.\d+)', text)
    if not m:
        m = re.search(r'val_bpb:(\d+\.\d+)', text)
    bpb = float(m.group(2) if m and m.lastindex >= 2 else (m.group(1) if m else '0'))
    steps = 0
    for sm in re.finditer(r'step:(\d+)/', text):
        steps = max(steps, int(sm.group(1)))
    if bpb > 0:
        results.append((name, bpb, steps))

# Sort by BPB
results.sort(key=lambda x: x[1])
control_bpbs = [bpb for n, bpb, s in results if 'control' in n]
control_mean = sum(control_bpbs) / len(control_bpbs) if control_bpbs else 0

print(f"{'Run':<25} {'BPB':>10} {'Delta':>8} {'Steps':>6}")
print('-' * 55)
for name, bpb, steps in results:
    delta = bpb - control_mean if control_mean else 0
    marker = ' ← BEST' if delta < -0.001 and 'control' not in name else ''
    print(f"{name:<25} {bpb:>10.4f} {delta:>+8.4f} {steps:>6}{marker}")

# Save to Drive
import shutil
for logfile in glob.glob('logs/geom_*.txt') + glob.glob('logs/cuda/geom_*.txt'):
    os.makedirs(f'{DRIVE_DIR}/ternary/logs', exist_ok=True)
    shutil.copy2(logfile, f'{DRIVE_DIR}/ternary/logs/{os.path.basename(logfile)}')

In [ ]:
# === 11. PHASE 1: TERNARY RESIDUAL + STRUCTURED ZEROS (~5 hours) ===
# Safest experiments — no arch change, no dim reduction, minimal compute overhead
%cd {WORK}

import os, torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 524288 if vram_gb >= 70 else (131072 if vram_gb >= 35 else 65536)

# Common training env vars (matching submission exactly)
BASE_ENV = (
    f'DATA_PATH=./data/datasets/fineweb10B_sp8192 '
    f'TOKENIZER_PATH=./data/tokenizers/fineweb_8192_bpe.model '
    f'VOCAB_SIZE=8192 '
    f'NUM_LAYERS=10 MODEL_DIM=768 NUM_HEADS=8 NUM_KV_HEADS=4 MLP_MULT=4 '
    f'EMBED_DIM=254 '
    f'BITNET_GROUP_SIZE=128 '
    f'ACTIVATION=relu2 '
    f'ROPE_TYPE=yarn YARN_MAX_LEN=2048 ROPE_BASE=5000 '
    f'LOGIT_SOFTCAP=10 SOFTCAP_TYPE=poly '
    f'QK_GAIN_INIT=2.25 '
    f'MATRIX_OPTIMIZER=muon MUON_BACKEND_STEPS=3 '
    f'MUON_MOMENTUM=0.95 MUON_MOMENTUM_WARMUP_START=0.85 MUON_MOMENTUM_WARMUP_STEPS=500 '
    f'MUON_WD=0.0 ADAM_WD=0.05 ADAM_LR=0.05 '
    f'MATRIX_LR=0.04 SCALAR_LR=0.02 TIED_EMBED_LR=0.02 HEAD_LR=0.02 '
    f'WARMDOWN_FRACTION=0.2 '
    f'FP_STORAGE=FP8 '
    f'TRAIN_BATCH_TOKENS={BATCH} TRAIN_SEQ_LEN=1024 '
    f'BIGRAM_HASH=0 SMEAR=0 DIFF_ATTN=0 MLP_GROUPS=0 MTP_HEADS=0 '
    f'TIE_EMBEDDINGS=1 '
    f'ITERATIONS=10000 MAX_WALLCLOCK_SECONDS=3600 '
    f'WARMUP_STEPS=5 VAL_LOSS_EVERY=500 TRAIN_LOG_EVERY=100 '
    f'USE_COMPILE=0 SEED=42 '
    f'GEOM_ALPHA=0 GEOM_BETA=0 GEOM_SIGNALS=geometric_field/g_signals.pt '
)

experiments = [
    # (name, extra env vars, description)
    ("phase1_control",    "", "Baseline control"),
    ("phase1_res_01_256", "RESIDUAL_EPSILON=0.1 RESIDUAL_COARSE_GS=256", "Residual eps=0.1, coarse=256"),
    ("phase1_res_01_512", "RESIDUAL_EPSILON=0.1 RESIDUAL_COARSE_GS=512", "Residual eps=0.1, coarse=512"),
    ("phase1_zeros_01",   "ZERO_BIAS_RANGE=0.1", "Structured zeros bias=0.1"),
    ("phase1_combo",      "RESIDUAL_EPSILON=0.1 RESIDUAL_COARSE_GS=512 ZERO_BIAS_RANGE=0.1", "Residual + zeros combined"),
]

for name, extra, desc in experiments:
    # Check for completed log
    log_candidates = [f'logs/cuda/{name}.txt', f'logs/{name}.txt']
    skip = False
    for lf in log_candidates:
        if os.path.exists(lf):
            with open(lf) as f:
                content = f.read()
            if 'stopping_early' in content or 'final_ternary' in content:
                print(f'SKIP {name} — already complete')
                skip = True
                break
    if skip:
        continue

    print(f'\n{"="*60}')
    print(f'{name}: {desc}')
    print(f'{"="*60}')

    !RUN_ID={name} {BASE_ENV} {extra} python3 train_gpt_cuda_ternary.py

In [ ]:
# === 12. COLLECT PHASE 1 RESULTS ===
%cd {WORK}
import re, glob, os

results = []
for logfile in sorted(glob.glob('logs/cuda/phase1_*.txt') + glob.glob('logs/phase1_*.txt')):
    name = os.path.basename(logfile).replace('.txt', '')
    with open(logfile) as f:
        text = f.read()
    # Find final roundtrip BPB
    m = re.search(r'final_ternary_roundtrip\s+val_loss:(\d+\.\d+)\s+val_bpb:(\d+\.\d+)', text)
    if m:
        bpb = float(m.group(2))
    else:
        # Fallback: last val_bpb
        matches = re.findall(r'val_bpb:(\d+\.\d+)', text)
        bpb = float(matches[-1]) if matches else 0
    # Pre-quant BPB
    m2 = re.search(r'step:\d+/\d+ val_loss:\d+\.\d+ val_bpb:(\d+\.\d+).*stopping', text)
    pre_quant = float(m2.group(1)) if m2 else 0
    # Step time
    m3 = re.search(r'avg:(\d+\.\d+)ms', text)
    step_ms = float(m3.group(1)) if m3 else 0
    steps = 0
    for sm in re.finditer(r'step:(\d+)/', text):
        steps = max(steps, int(sm.group(1)))
    if bpb > 0:
        results.append((name, bpb, pre_quant, step_ms, steps))

# Sort by BPB
results.sort(key=lambda x: x[1])
control_bpb = [b for n, b, _, _, _ in results if 'control' in n]
ctrl_mean = sum(control_bpb) / len(control_bpb) if control_bpb else 0

print(f"{'Run':<25} {'RT BPB':>8} {'PreQ':>8} {'QTax':>7} {'Delta':>7} {'ms/step':>7} {'Steps':>6}")
print('-' * 75)
for name, bpb, pre_q, ms, steps in results:
    delta = bpb - ctrl_mean if ctrl_mean else 0
    qtax = bpb - pre_q if pre_q else 0
    marker = ' ←' if delta < -0.001 and 'control' not in name else ''
    print(f"{name:<25} {bpb:>8.4f} {pre_q:>8.4f} {qtax:>+7.4f} {delta:>+7.4f} {ms:>7.1f} {steps:>6}{marker}")

# Save to Drive
import shutil
DRIVE_DIR = '/content/drive/MyDrive/parameter-golf'
for logfile in glob.glob('logs/cuda/phase1_*.txt') + glob.glob('logs/phase1_*.txt'):
    os.makedirs(f'{DRIVE_DIR}/ternary/logs', exist_ok=True)
    shutil.copy2(logfile, f'{DRIVE_DIR}/ternary/logs/{os.path.basename(logfile)}')

In [ ]:
# === 13. ACTIVATION EXPERIMENTS (~6 hours) ===
# Exp 1: Parametric power (relu^p, p learned per layer)
# Exp 2: Stochastic depth (random block skipping, free speedup)
# Exp 3: GaugeReLU (phase-magnitude activation)
%cd {WORK}

import os, torch

vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
BATCH = 524288 if vram_gb >= 70 else (131072 if vram_gb >= 35 else 65536)

BASE_ENV = (
    f'DATA_PATH=./data/datasets/fineweb10B_sp8192 '
    f'TOKENIZER_PATH=./data/tokenizers/fineweb_8192_bpe.model '
    f'VOCAB_SIZE=8192 '
    f'NUM_LAYERS=10 MODEL_DIM=768 NUM_HEADS=8 NUM_KV_HEADS=4 MLP_MULT=4 '
    f'EMBED_DIM=254 BITNET_GROUP_SIZE=128 ACTIVATION=relu2 '
    f'ROPE_TYPE=yarn YARN_MAX_LEN=2048 ROPE_BASE=5000 '
    f'LOGIT_SOFTCAP=10 SOFTCAP_TYPE=poly QK_GAIN_INIT=2.25 '
    f'MATRIX_OPTIMIZER=muon MUON_BACKEND_STEPS=3 '
    f'MUON_MOMENTUM=0.95 MUON_MOMENTUM_WARMUP_START=0.85 MUON_MOMENTUM_WARMUP_STEPS=500 '
    f'MUON_WD=0.0 ADAM_WD=0.05 ADAM_LR=0.05 '
    f'MATRIX_LR=0.04 SCALAR_LR=0.02 TIED_EMBED_LR=0.02 HEAD_LR=0.02 '
    f'WARMDOWN_FRACTION=0.2 FP_STORAGE=FP8 '
    f'TRAIN_BATCH_TOKENS={BATCH} TRAIN_SEQ_LEN=1024 '
    f'BIGRAM_HASH=0 SMEAR=0 DIFF_ATTN=0 MLP_GROUPS=0 MTP_HEADS=0 TIE_EMBEDDINGS=1 '
    f'ITERATIONS=10000 MAX_WALLCLOCK_SECONDS=3600 '
    f'WARMUP_STEPS=5 VAL_LOSS_EVERY=500 TRAIN_LOG_EVERY=100 '
    f'USE_COMPILE=0 SEED=42 '
    f'GEOM_ALPHA=0 GEOM_BETA=0 RESIDUAL_EPSILON=0 ZERO_BIAS_RANGE=0 '
)

experiments = [
    # (name, extra env vars, description)
    ("act_control",     "POWER_ACT=0 STOCH_DEPTH=0 GAUGE_RELU=0",
     "Control baseline"),
    ("act_power",       "POWER_ACT=1 STOCH_DEPTH=0 GAUGE_RELU=0",
     "Parametric power (p learned per layer, init=2.0)"),
    ("act_stoch_02",    "POWER_ACT=0 STOCH_DEPTH=0.2 GAUGE_RELU=0",
     "Stochastic depth (max drop=0.2, skip first/last 2)"),
    ("act_gauge",       "POWER_ACT=0 STOCH_DEPTH=0 GAUGE_RELU=1",
     "GaugeReLU (phase-magnitude activation)"),
    ("act_power_stoch", "POWER_ACT=1 STOCH_DEPTH=0.2 GAUGE_RELU=0",
     "Parametric power + stochastic depth (stacked)"),
    ("act_gauge_stoch", "POWER_ACT=0 STOCH_DEPTH=0.2 GAUGE_RELU=1",
     "GaugeReLU + stochastic depth (stacked)"),
]

for name, extra, desc in experiments:
    log_candidates = [f'logs/cuda/{name}.txt', f'logs/{name}.txt']
    skip = False
    for lf in log_candidates:
        if os.path.exists(lf):
            with open(lf) as f:
                content = f.read()
            # Only skip if we see actual training output (not just source code dump)
            if 'final_ternary_roundtrip' in content:
                print(f'SKIP {name} — already complete')
                skip = True
                break
    if skip:
        continue

    # Delete any stale crash logs
    for lf in log_candidates:
        if os.path.exists(lf):
            os.remove(lf)

    print(f'\n{"="*60}')
    print(f'{name}: {desc}')
    print(f'{"="*60}')

    !RUN_ID={name} {BASE_ENV} {extra} python3 train_gpt_cuda_ternary.py

In [ ]:
# === 14. COLLECT ACTIVATION EXPERIMENT RESULTS ===
%cd {WORK}
import re, glob, os

results = []
for logfile in sorted(glob.glob('logs/cuda/act_*.txt') + glob.glob('logs/act_*.txt')):
    name = os.path.basename(logfile).replace('.txt', '')
    with open(logfile) as f:
        text = f.read()
    m = re.search(r'final_ternary_roundtrip\s+val_loss:(\d+\.\d+)\s+val_bpb:(\d+\.\d+)', text)
    rt_bpb = float(m.group(2)) if m else 0
    # Last val_bpb before stopping
    matches = re.findall(r'val_bpb:(\d+\.\d+)', text)
    pre_bpb = float(matches[-2]) if len(matches) >= 2 and rt_bpb > 0 else 0
    m3 = re.search(r'avg:(\d+\.\d+)ms', text)
    step_ms = float(m3.group(1)) if m3 else 0
    steps = max([int(s) for s in re.findall(r'step:(\d+)/', text)], default=0)
    # Check for learned power values
    powers = re.findall(r'power_layer_\d+=(\d+\.\d+)', text)
    if rt_bpb > 0:
        results.append((name, rt_bpb, pre_bpb, step_ms, steps, powers))

results.sort(key=lambda x: x[1])
ctrl = [b for n, b, _, _, _, _ in results if 'control' in n]
ctrl_mean = sum(ctrl) / len(ctrl) if ctrl else 0

print(f"{'Run':<22} {'RT BPB':>8} {'PreQ':>8} {'QTax':>7} {'Delta':>7} {'ms/st':>6} {'Steps':>5}")
print('-' * 72)
for name, bpb, pre, ms, steps, powers in results:
    delta = bpb - ctrl_mean if ctrl_mean else 0
    qtax = bpb - pre if pre else 0
    marker = ' ←' if delta < -0.001 and 'control' not in name else ''
    print(f"{name:<22} {bpb:>8.4f} {pre:>8.4f} {qtax:>+7.4f} {delta:>+7.4f} {ms:>6.0f} {steps:>5}{marker}")
    if powers:
        print(f"  {'Learned powers:':<20} {', '.join(f'{float(p):.2f}' for p in powers[:5])}...")

import shutil
DRIVE_DIR = '/content/drive/MyDrive/parameter-golf'
for logfile in glob.glob('logs/cuda/act_*.txt') + glob.glob('logs/act_*.txt'):
    os.makedirs(f'{DRIVE_DIR}/ternary/logs', exist_ok=True)
    shutil.copy2(logfile, f'{DRIVE_DIR}/ternary/logs/{os.path.basename(logfile)}')